In [7]:
import os
os.getcwd()

'D:\\Vtech\\vibeRank\\VibeRank\\src\\viberank'

In [8]:
%cd "D:\Vtech\vibeRank\VibeRank\src\viberank"

D:\Vtech\vibeRank\VibeRank\src\viberank


In [9]:
from evaluation.count_cycles import GraphTools

now we need code to load the data files and count cycles. What does the data even look like?



In [10]:
import os
import pandas as pd


In [16]:
base_dir= 'D:\\Vtech\\vibeRank\\responses'
dataset = 'VISPDAT'
t_path = os.path.join(base_dir,dataset,'rc_responses')

parsed_files = [
    'QWEN_vispdat_20260428_133227.csv', 'ParseLLAMA_vispdat_20260428_184821.csv', 'Parseddeepseek8B_vispdat_20260428_203611.csv'
]





In [18]:
df_parsed = pd.read_csv(os.path.join(t_path, parsed_files[0]))
os.path.join(t_path, parsed_files[0])

'D:\\Vtech\\vibeRank\\responses\\VISPDAT\\rc_responses\\QWEN_vispdat_20260428_133227.csv'

In [24]:
df_parsed.head()

,left_item,right_item,more_vulnerable_household
0,262923,303448,Household 1
1,262923,303448,Household 1
2,262923,303448,Household 1
3,262923,303448,Household 1
4,262923,303448,Household 1


In [31]:
df_parsed = df_parsed[['left_item','right_item','more_vulnerable_household']]

In [32]:
df_parsed[
    (df_parsed["left_item"] == 262923) &
    (df_parsed["right_item"] == 327739)
]

,left_item,right_item,more_vulnerable_household
3100,262923,327739,indeterminate
3101,262923,327739,indeterminate
3102,262923,327739,indeterminate
3103,262923,327739,indeterminate
3104,262923,327739,indeterminate
3105,262923,327739,indeterminate
3106,262923,327739,indeterminate
3107,262923,327739,Household 1
3108,262923,327739,Household 1
3109,262923,327739,Household 1


In [34]:
df_parsed[
    (df_parsed["left_item"] == 337899  ) &
    (df_parsed["right_item"] == 218687)
]

,left_item,right_item,more_vulnerable_household
3190,337899,218687,Household 2
3191,337899,218687,Household 1
3192,337899,218687,Household 1
3193,337899,218687,Household 2
3194,337899,218687,Household 1
3195,337899,218687,Household 1
3196,337899,218687,Household 1
3197,337899,218687,Household 2
3198,337899,218687,Household 2
3199,337899,218687,Household 2


find majority winner

In [26]:
def normalize_winner(x):
    x = str(x).strip().lower()

    if x in ["household1", "household 1", "1", "left"]:
        return "household1"
    elif x in ["household2", "household 2", "2", "right"]:
        return "household2"
    elif x in ["indeterminate", "inderminate", "inderterminate", "unknown", "tie", "nan", "none"]:
        return "indeterminate"
    else:
        return x


df_tmp = df_parsed.copy()
df_tmp["winner_clean"] = df_tmp["more_vulnerable_household"].apply(normalize_winner)


def choose_majority_winner(group):
    total_votes = len(group)

    counts = group["winner_clean"].value_counts(dropna=False)

    h1_votes = counts.get("household1", 0)
    h2_votes = counts.get("household2", 0)
    ind_votes = counts.get("indeterminate", 0)

    # indeterminate can only win if ALL comparisons are indeterminate
    if ind_votes == total_votes:
        majority_winner = "indeterminate"
        majority_votes = ind_votes

    else:
        # otherwise ignore indeterminate and choose between household1 and household2
        if h1_votes > h2_votes:
            majority_winner = "household1"
            majority_votes = h1_votes
        elif h2_votes > h1_votes:
            majority_winner = "household2"
            majority_votes = h2_votes
        else:
            # rare case: household1 and household2 tie after removing indeterminate
            majority_winner = "tie_household1_household2"
            majority_votes = h1_votes

    return pd.Series({
        "majority_winner": majority_winner,
        "majority_votes": majority_votes,
        "total_votes": total_votes,
        "household1_votes": h1_votes,
        "household2_votes": h2_votes,
        "indeterminate_votes": ind_votes,
    })


df_majority = (
    df_tmp
    .groupby(["left_item", "right_item"])
    .apply(choose_majority_winner)
    .reset_index()
)

df_majority

C:\Users\ShafkatFarabi\AppData\Local\Temp\ipykernel_3840\2311459334.py:58: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(choose_majority_winner)


,left_item,right_item,majority_winner,majority_votes,total_votes,household1_votes,household2_votes,indeterminate_votes
0,18,218687,household2,10,10,0,10,0
1,18,230809,household2,10,10,0,10,0
2,18,325477,household2,10,10,0,10,0
3,18,349157,household2,10,10,0,10,0
4,505,18,household2,10,10,0,10,0
...,...,...,...,...,...,...,...,...
430,353058,331859,household1,10,10,10,0,0
431,353058,337899,household2,10,10,0,10,0
432,353058,341645,household1,10,10,10,0,0
433,353058,349157,household2,10,10,0,10,0


In [29]:
# we need to transform this to a list of edges.
# point towards winner
# if winner household1 =>   left_item <- right_item
# if winner household2 =>   right_item -> left_item


edges = []

for row in df_majority.itertuples(index=False):
    left = row.left_item
    right = row.right_item
    winner = str(row.majority_winner).strip().lower()

    if winner in ["household1", "household 1", "1", "left"]:
        # left wins, so right -> left
        edges.append((right, left))

    elif winner in ["household2", "household 2", "2", "right"]:
        # right wins, so left -> right
        edges.append((left, right))

    else:
        print("Unknown winner:",left,right, row.majority_winner)

len(edges), edges[:10]





Unknown winner: 337899 218687 tie_household1_household2


(434,
 [(18, 218687),
  (18, 230809),
  (18, 325477),
  (18, 349157),
  (505, 18),
  (505, 207017),
  (505, 218687),
  (505, 230809),
  (266515, 505),
  (289226, 505)])

In [22]:
gt = GraphTools(edges)

print("Complete graph check:", gt.validate_complete_graph())

cnt_formula = gt.count_triads()
cnt_naive, cycles = gt.naive_count_triads()

print("Formula cycle count:", cnt_formula)
print("Naive cycle count:", cnt_naive)


Complete graph check: failed
Formula cycle count: 75
Naive cycle count: 48


In [15]:
len(gt.nodes)

30

In [14]:
from collections import Counter

def diagnose_edges(edges):
    nodes = set()
    unordered_pair_counts = Counter()
    directed_pair_counts = Counter()
    self_loops = []

    for u, v in edges:
        nodes.add(u)
        nodes.add(v)

        directed_pair_counts[(u, v)] += 1

        if u == v:
            self_loops.append((u, v))
        else:
            unordered_pair_counts[tuple(sorted([u, v]))] += 1

    n = len(nodes)
    expected_pairs = n * (n - 1) // 2
    actual_unordered_pairs = len(unordered_pair_counts)

    missing_pairs = []
    duplicate_unordered_pairs = []

    nodes_list = sorted(nodes)

    for i in range(len(nodes_list)):
        for j in range(i + 1, len(nodes_list)):
            pair = (nodes_list[i], nodes_list[j])
            cnt = unordered_pair_counts.get(pair, 0)

            if cnt == 0:
                missing_pairs.append(pair)
            elif cnt > 1:
                duplicate_unordered_pairs.append((pair, cnt))

    duplicate_directed_edges = [
        (pair, cnt)
        for pair, cnt in directed_pair_counts.items()
        if cnt > 1
    ]

    print("num nodes:", n)
    print("num directed edges:", len(edges))
    print("expected unordered pairs:", expected_pairs)
    print("actual unordered pairs:", actual_unordered_pairs)
    print("self loops:", len(self_loops))
    print("missing unordered pairs:", len(missing_pairs))
    print("duplicate unordered pairs:", len(duplicate_unordered_pairs))
    print("duplicate directed edges:", len(duplicate_directed_edges))

    print("\nFirst few missing pairs:")
    print(missing_pairs[:10])

    print("\nFirst few duplicate unordered pairs:")
    print(duplicate_unordered_pairs[:10])

    print("\nFirst few duplicate directed edges:")
    print(duplicate_directed_edges[:10])

diagnose_edges(edges)

num nodes: 30
num directed edges: 434
expected unordered pairs: 435
actual unordered pairs: 434
self loops: 0
missing unordered pairs: 1
duplicate unordered pairs: 0
duplicate directed edges: 0

First few missing pairs:
[(262923, 327739)]

First few duplicate unordered pairs:
[]

First few duplicate directed edges:
[]


In [4]:
import os
import pandas as pd
from collections import Counter
from evaluation.count_cycles import GraphTools

base_dir = r"D:\Vtech\vibeRank\responses"
dataset = "VISPDAT"
t_path = os.path.join(base_dir, dataset, "rc_responses")

parsed_files = [
    "QWEN_vispdat_20260428_133227.csv",
    "ParseLLAMA_vispdat_20260428_184821.csv",
    "Parseddeepseek8B_vispdat_20260428_203611.csv",
]


def canonical_pair(u, v):
    # robust even if node ids are mixed int/string
    return tuple(sorted([u, v], key=lambda x: str(x)))


def diagnose_edges(edges, verbose=False):
    nodes = set()
    unordered_pair_counts = Counter()
    directed_pair_counts = Counter()
    self_loops = []

    for u, v in edges:
        nodes.add(u)
        nodes.add(v)

        directed_pair_counts[(u, v)] += 1

        if u == v:
            self_loops.append((u, v))
        else:
            unordered_pair_counts[canonical_pair(u, v)] += 1

    n = len(nodes)
    expected_edges = n * (n - 1) // 2
    actual_edges = len(unordered_pair_counts)

    missing_pairs = []
    duplicate_unordered_pairs = []

    nodes_list = sorted(nodes, key=lambda x: str(x))

    for i in range(len(nodes_list)):
        for j in range(i + 1, len(nodes_list)):
            pair = canonical_pair(nodes_list[i], nodes_list[j])
            cnt = unordered_pair_counts.get(pair, 0)

            if cnt == 0:
                missing_pairs.append(pair)
            elif cnt > 1:
                duplicate_unordered_pairs.append((pair, cnt))

    duplicate_directed_edges = [
        (pair, cnt)
        for pair, cnt in directed_pair_counts.items()
        if cnt > 1
    ]

    if verbose:
        print("num nodes:", n)
        print("num directed edges:", len(edges))
        print("expected unordered edges:", expected_edges)
        print("actual unordered edges:", actual_edges)
        print("self loops:", len(self_loops))
        print("missing unordered pairs:", len(missing_pairs))
        print("duplicate unordered pairs:", len(duplicate_unordered_pairs))
        print("duplicate directed edges:", len(duplicate_directed_edges))
        print("\nFirst few missing pairs:")
        print(missing_pairs[:10])
        print("\nFirst few duplicate unordered pairs:")
        print(duplicate_unordered_pairs[:10])
        print("\nFirst few duplicate directed edges:")
        print(duplicate_directed_edges[:10])

    return {
        "num_nodes": n,
        "num_directed_edges": len(edges),
        "expected_edges": expected_edges,
        "actual_edges": actual_edges,
        "num_self_loops": len(self_loops),
        "num_missing_edges": len(missing_pairs),
        "missing_edges": missing_pairs,
        "num_duplicate_unordered_edges": len(duplicate_unordered_pairs),
        "duplicate_unordered_edges": duplicate_unordered_pairs,
        "num_duplicate_directed_edges": len(duplicate_directed_edges),
        "duplicate_directed_edges": duplicate_directed_edges,
    }


def build_majority_edges_from_file(file_name, verbose=False):
    file_path = os.path.join(t_path, file_name)

    df_parsed = pd.read_csv(file_path)
    df_parsed = df_parsed[["left_item", "right_item", "more_vulnerable_household"]].copy()

    # Count votes per ordered left/right pair and winner
    vote_counts = (
        df_parsed
        .groupby(["left_item", "right_item", "more_vulnerable_household"])
        .size()
        .reset_index(name="votes")
    )

    # For each left/right pair, keep the winner with max votes
    df_majority = (
        vote_counts
        .sort_values(
            ["left_item", "right_item", "votes"],
            ascending=[True, True, False]
        )
        .drop_duplicates(["left_item", "right_item"], keep="first")
        .rename(columns={"more_vulnerable_household": "majority_winner"})
        .reset_index(drop=True)
    )

    edges = []
    unknown_winners = []

    for row in df_majority.itertuples(index=False):
        left = row.left_item
        right = row.right_item
        winner = str(row.majority_winner).strip().lower()

        if winner in ["household1", "household 1", "1", "left"]:
            # left wins, so edge points right -> left
            edges.append((right, left))

        elif winner in ["household2", "household 2", "2", "right"]:
            # right wins, so edge points left -> right
            edges.append((left, right))

        else:
            unknown_winners.append(row.majority_winner)

    gt = GraphTools(edges)

    complete_check = gt.validate_complete_graph()

    try:
        formula_count = gt.count_triads()
    except Exception as e:
        formula_count = None
        if verbose:
            print("Formula count failed:", e)

    try:
        naive_count, cycles = gt.naive_count_triads()
    except Exception as e:
        naive_count = None
        cycles = None
        if verbose:
            print("Naive count failed:", e)

    diag = diagnose_edges(edges, verbose=verbose)

    result = {
        "file": file_name,
        "complete_check": complete_check,
        "naive_count": naive_count,
        "formula_count": formula_count,
        "num_nodes": diag["num_nodes"],
        "expected_edges": diag["expected_edges"],
        "actual_edges": diag["actual_edges"],
        "num_missing_edges": diag["num_missing_edges"],
        "missing_edges": diag["missing_edges"],
        "num_duplicate_unordered_edges": diag["num_duplicate_unordered_edges"],
        "num_duplicate_directed_edges": diag["num_duplicate_directed_edges"],
        "num_unknown_winners": len(unknown_winners),
        "unknown_winners": unknown_winners,
    }

    return result, df_majority, edges, cycles


all_results = []
all_majority_dfs = {}
all_edges = {}
all_cycles = {}

for file_name in parsed_files:
    print("=" * 100)
    print("Processing:", file_name)

    result, df_majority, edges, cycles = build_majority_edges_from_file(
        file_name=file_name,
        verbose=True
    )

    all_results.append(result)
    all_majority_dfs[file_name] = df_majority
    all_edges[file_name] = edges
    all_cycles[file_name] = cycles

summary_df = pd.DataFrame(all_results)

summary_df

Processing: QWEN_vispdat_20260428_133227.csv
num nodes: 30
num directed edges: 434
expected unordered edges: 435
actual unordered edges: 434
self loops: 0
missing unordered pairs: 1
duplicate unordered pairs: 0
duplicate directed edges: 0

First few missing pairs:
[(262923, 327739)]

First few duplicate unordered pairs:
[]

First few duplicate directed edges:
[]
Processing: ParseLLAMA_vispdat_20260428_184821.csv
num nodes: 30
num directed edges: 435
expected unordered edges: 435
actual unordered edges: 435
self loops: 0
missing unordered pairs: 0
duplicate unordered pairs: 0
duplicate directed edges: 0

First few missing pairs:
[]

First few duplicate unordered pairs:
[]

First few duplicate directed edges:
[]
Processing: Parseddeepseek8B_vispdat_20260428_203611.csv
num nodes: 30
num directed edges: 435
expected unordered edges: 435
actual unordered edges: 435
self loops: 0
missing unordered pairs: 0
duplicate unordered pairs: 0
duplicate directed edges: 0

First few missing pairs:
[]


,file,complete_check,naive_count,formula_count,num_nodes,expected_edges,actual_edges,num_missing_edges,missing_edges,num_duplicate_unordered_edges,num_duplicate_directed_edges,num_unknown_winners,unknown_winners
0,QWEN_vispdat_20260428_133227.csv,failed,48,75,30,435,434,1,"[(262923, 327739)]",0,0,1,[indeterminate]
1,ParseLLAMA_vispdat_20260428_184821.csv,passed,30,30,30,435,435,0,[],0,0,0,[]
2,Parseddeepseek8B_vispdat_20260428_203611.csv,passed,89,89,30,435,435,0,[],0,0,0,[]


In [5]:
import os
import pandas as pd
from collections import Counter
from evaluation.count_cycles import GraphTools

base_dir = r"D:\Vtech\vibeRank\responses"
dataset = "TAYVISPDAT"
t_path = os.path.join(base_dir, dataset, "rc_responses")

parsed_files = [
    "ParsedQWEN_TAY.csv",
    "llama_TAY_parsed.csv",
    "DS_TAY_parsed.csv",
]


def canonical_pair(u, v):
    # robust even if node ids are mixed int/string
    return tuple(sorted([u, v], key=lambda x: str(x)))


def diagnose_edges(edges, verbose=False):
    nodes = set()
    unordered_pair_counts = Counter()
    directed_pair_counts = Counter()
    self_loops = []

    for u, v in edges:
        nodes.add(u)
        nodes.add(v)

        directed_pair_counts[(u, v)] += 1

        if u == v:
            self_loops.append((u, v))
        else:
            unordered_pair_counts[canonical_pair(u, v)] += 1

    n = len(nodes)
    expected_edges = n * (n - 1) // 2
    actual_edges = len(unordered_pair_counts)

    missing_pairs = []
    duplicate_unordered_pairs = []

    nodes_list = sorted(nodes, key=lambda x: str(x))

    for i in range(len(nodes_list)):
        for j in range(i + 1, len(nodes_list)):
            pair = canonical_pair(nodes_list[i], nodes_list[j])
            cnt = unordered_pair_counts.get(pair, 0)

            if cnt == 0:
                missing_pairs.append(pair)
            elif cnt > 1:
                duplicate_unordered_pairs.append((pair, cnt))

    duplicate_directed_edges = [
        (pair, cnt)
        for pair, cnt in directed_pair_counts.items()
        if cnt > 1
    ]

    if verbose:
        print("num nodes:", n)
        print("num directed edges:", len(edges))
        print("expected unordered edges:", expected_edges)
        print("actual unordered edges:", actual_edges)
        print("self loops:", len(self_loops))
        print("missing unordered pairs:", len(missing_pairs))
        print("duplicate unordered pairs:", len(duplicate_unordered_pairs))
        print("duplicate directed edges:", len(duplicate_directed_edges))
        print("\nFirst few missing pairs:")
        print(missing_pairs[:10])
        print("\nFirst few duplicate unordered pairs:")
        print(duplicate_unordered_pairs[:10])
        print("\nFirst few duplicate directed edges:")
        print(duplicate_directed_edges[:10])

    return {
        "num_nodes": n,
        "num_directed_edges": len(edges),
        "expected_edges": expected_edges,
        "actual_edges": actual_edges,
        "num_self_loops": len(self_loops),
        "num_missing_edges": len(missing_pairs),
        "missing_edges": missing_pairs,
        "num_duplicate_unordered_edges": len(duplicate_unordered_pairs),
        "duplicate_unordered_edges": duplicate_unordered_pairs,
        "num_duplicate_directed_edges": len(duplicate_directed_edges),
        "duplicate_directed_edges": duplicate_directed_edges,
    }


def build_majority_edges_from_file(file_name, verbose=False):
    file_path = os.path.join(t_path, file_name)

    df_parsed = pd.read_csv(file_path)
    df_parsed = df_parsed[["left_item", "right_item", "more_vulnerable_household"]].copy()

    # Count votes per ordered left/right pair and winner
    vote_counts = (
        df_parsed
        .groupby(["left_item", "right_item", "more_vulnerable_household"])
        .size()
        .reset_index(name="votes")
    )

    # For each left/right pair, keep the winner with max votes
    df_majority = (
        vote_counts
        .sort_values(
            ["left_item", "right_item", "votes"],
            ascending=[True, True, False]
        )
        .drop_duplicates(["left_item", "right_item"], keep="first")
        .rename(columns={"more_vulnerable_household": "majority_winner"})
        .reset_index(drop=True)
    )

    edges = []
    unknown_winners = []

    for row in df_majority.itertuples(index=False):
        left = row.left_item
        right = row.right_item
        winner = str(row.majority_winner).strip().lower()

        if winner in ["household1", "household 1", "1", "left"]:
            # left wins, so edge points right -> left
            edges.append((right, left))

        elif winner in ["household2", "household 2", "2", "right"]:
            # right wins, so edge points left -> right
            edges.append((left, right))

        else:
            unknown_winners.append(row.majority_winner)

    gt = GraphTools(edges)

    complete_check = gt.validate_complete_graph()

    try:
        formula_count = gt.count_triads()
    except Exception as e:
        formula_count = None
        if verbose:
            print("Formula count failed:", e)

    try:
        naive_count, cycles = gt.naive_count_triads()
    except Exception as e:
        naive_count = None
        cycles = None
        if verbose:
            print("Naive count failed:", e)

    diag = diagnose_edges(edges, verbose=verbose)

    result = {
        "file": file_name,
        "complete_check": complete_check,
        "naive_count": naive_count,
        "formula_count": formula_count,
        "num_nodes": diag["num_nodes"],
        "expected_edges": diag["expected_edges"],
        "actual_edges": diag["actual_edges"],
        "num_missing_edges": diag["num_missing_edges"],
        "missing_edges": diag["missing_edges"],
        "num_duplicate_unordered_edges": diag["num_duplicate_unordered_edges"],
        "num_duplicate_directed_edges": diag["num_duplicate_directed_edges"],
        "num_unknown_winners": len(unknown_winners),
        "unknown_winners": unknown_winners,
    }

    return result, df_majority, edges, cycles


all_results = []
all_majority_dfs = {}
all_edges = {}
all_cycles = {}

for file_name in parsed_files:
    print("=" * 100)
    print("Processing:", file_name)

    result, df_majority, edges, cycles = build_majority_edges_from_file(
        file_name=file_name,
        verbose=True
    )

    all_results.append(result)
    all_majority_dfs[file_name] = df_majority
    all_edges[file_name] = edges
    all_cycles[file_name] = cycles

summary_df = pd.DataFrame(all_results)

summary_df

Processing: ParsedQWEN_TAY.csv
num nodes: 30
num directed edges: 434
expected unordered edges: 435
actual unordered edges: 434
self loops: 0
missing unordered pairs: 1
duplicate unordered pairs: 0
duplicate directed edges: 0

First few missing pairs:
[(261953, 338384)]

First few duplicate unordered pairs:
[]

First few duplicate directed edges:
[]
Processing: llama_TAY_parsed.csv
num nodes: 30
num directed edges: 435
expected unordered edges: 435
actual unordered edges: 435
self loops: 0
missing unordered pairs: 0
duplicate unordered pairs: 0
duplicate directed edges: 0

First few missing pairs:
[]

First few duplicate unordered pairs:
[]

First few duplicate directed edges:
[]
Processing: DS_TAY_parsed.csv
num nodes: 30
num directed edges: 435
expected unordered edges: 435
actual unordered edges: 435
self loops: 0
missing unordered pairs: 0
duplicate unordered pairs: 0
duplicate directed edges: 0

First few missing pairs:
[]

First few duplicate unordered pairs:
[]

First few duplica

,file,complete_check,naive_count,formula_count,num_nodes,expected_edges,actual_edges,num_missing_edges,missing_edges,num_duplicate_unordered_edges,num_duplicate_directed_edges,num_unknown_winners,unknown_winners
0,ParsedQWEN_TAY.csv,failed,97,111,30,435,434,1,"[(261953, 338384)]",0,0,1,[indeterminate]
1,llama_TAY_parsed.csv,passed,47,47,30,435,435,0,[],0,0,0,[]
2,DS_TAY_parsed.csv,passed,130,130,30,435,435,0,[],0,0,0,[]


In [6]:
import os
import pandas as pd
from collections import Counter
from evaluation.count_cycles import GraphTools

base_dir = r"D:\Vtech\vibeRank\responses"
dataset = "VIFSPDAT"
t_path = os.path.join(base_dir, dataset, "rc_responses")

parsed_files = [
    "ParsedQWEN_vispdat.csv",
    "ParseLLAMA_vifspdat.csv",
    "Parseddeepseek8B_vifspdat.csv",
]


def canonical_pair(u, v):
    # robust even if node ids are mixed int/string
    return tuple(sorted([u, v], key=lambda x: str(x)))


def diagnose_edges(edges, verbose=False):
    nodes = set()
    unordered_pair_counts = Counter()
    directed_pair_counts = Counter()
    self_loops = []

    for u, v in edges:
        nodes.add(u)
        nodes.add(v)

        directed_pair_counts[(u, v)] += 1

        if u == v:
            self_loops.append((u, v))
        else:
            unordered_pair_counts[canonical_pair(u, v)] += 1

    n = len(nodes)
    expected_edges = n * (n - 1) // 2
    actual_edges = len(unordered_pair_counts)

    missing_pairs = []
    duplicate_unordered_pairs = []

    nodes_list = sorted(nodes, key=lambda x: str(x))

    for i in range(len(nodes_list)):
        for j in range(i + 1, len(nodes_list)):
            pair = canonical_pair(nodes_list[i], nodes_list[j])
            cnt = unordered_pair_counts.get(pair, 0)

            if cnt == 0:
                missing_pairs.append(pair)
            elif cnt > 1:
                duplicate_unordered_pairs.append((pair, cnt))

    duplicate_directed_edges = [
        (pair, cnt)
        for pair, cnt in directed_pair_counts.items()
        if cnt > 1
    ]

    if verbose:
        print("num nodes:", n)
        print("num directed edges:", len(edges))
        print("expected unordered edges:", expected_edges)
        print("actual unordered edges:", actual_edges)
        print("self loops:", len(self_loops))
        print("missing unordered pairs:", len(missing_pairs))
        print("duplicate unordered pairs:", len(duplicate_unordered_pairs))
        print("duplicate directed edges:", len(duplicate_directed_edges))
        print("\nFirst few missing pairs:")
        print(missing_pairs[:10])
        print("\nFirst few duplicate unordered pairs:")
        print(duplicate_unordered_pairs[:10])
        print("\nFirst few duplicate directed edges:")
        print(duplicate_directed_edges[:10])

    return {
        "num_nodes": n,
        "num_directed_edges": len(edges),
        "expected_edges": expected_edges,
        "actual_edges": actual_edges,
        "num_self_loops": len(self_loops),
        "num_missing_edges": len(missing_pairs),
        "missing_edges": missing_pairs,
        "num_duplicate_unordered_edges": len(duplicate_unordered_pairs),
        "duplicate_unordered_edges": duplicate_unordered_pairs,
        "num_duplicate_directed_edges": len(duplicate_directed_edges),
        "duplicate_directed_edges": duplicate_directed_edges,
    }


def build_majority_edges_from_file(file_name, verbose=False):
    file_path = os.path.join(t_path, file_name)

    df_parsed = pd.read_csv(file_path)
    df_parsed = df_parsed[["left_item", "right_item", "more_vulnerable_household"]].copy()

    # Count votes per ordered left/right pair and winner
    vote_counts = (
        df_parsed
        .groupby(["left_item", "right_item", "more_vulnerable_household"])
        .size()
        .reset_index(name="votes")
    )

    # For each left/right pair, keep the winner with max votes
    df_majority = (
        vote_counts
        .sort_values(
            ["left_item", "right_item", "votes"],
            ascending=[True, True, False]
        )
        .drop_duplicates(["left_item", "right_item"], keep="first")
        .rename(columns={"more_vulnerable_household": "majority_winner"})
        .reset_index(drop=True)
    )

    edges = []
    unknown_winners = []

    for row in df_majority.itertuples(index=False):
        left = row.left_item
        right = row.right_item
        winner = str(row.majority_winner).strip().lower()

        if winner in ["household1", "household 1", "1", "left"]:
            # left wins, so edge points right -> left
            edges.append((right, left))

        elif winner in ["household2", "household 2", "2", "right"]:
            # right wins, so edge points left -> right
            edges.append((left, right))

        else:
            unknown_winners.append(row.majority_winner)

    gt = GraphTools(edges)

    complete_check = gt.validate_complete_graph()

    try:
        formula_count = gt.count_triads()
    except Exception as e:
        formula_count = None
        if verbose:
            print("Formula count failed:", e)

    try:
        naive_count, cycles = gt.naive_count_triads()
    except Exception as e:
        naive_count = None
        cycles = None
        if verbose:
            print("Naive count failed:", e)

    diag = diagnose_edges(edges, verbose=verbose)

    result = {
        "file": file_name,
        "complete_check": complete_check,
        "naive_count": naive_count,
        "formula_count": formula_count,
        "num_nodes": diag["num_nodes"],
        "expected_edges": diag["expected_edges"],
        "actual_edges": diag["actual_edges"],
        "num_missing_edges": diag["num_missing_edges"],
        "missing_edges": diag["missing_edges"],
        "num_duplicate_unordered_edges": diag["num_duplicate_unordered_edges"],
        "num_duplicate_directed_edges": diag["num_duplicate_directed_edges"],
        "num_unknown_winners": len(unknown_winners),
        "unknown_winners": unknown_winners,
    }

    return result, df_majority, edges, cycles


all_results = []
all_majority_dfs = {}
all_edges = {}
all_cycles = {}

for file_name in parsed_files:
    print("=" * 100)
    print("Processing:", file_name)

    result, df_majority, edges, cycles = build_majority_edges_from_file(
        file_name=file_name,
        verbose=True
    )

    all_results.append(result)
    all_majority_dfs[file_name] = df_majority
    all_edges[file_name] = edges
    all_cycles[file_name] = cycles

summary_df = pd.DataFrame(all_results)

summary_df

Processing: ParsedQWEN_vispdat.csv
num nodes: 30
num directed edges: 435
expected unordered edges: 435
actual unordered edges: 435
self loops: 0
missing unordered pairs: 0
duplicate unordered pairs: 0
duplicate directed edges: 0

First few missing pairs:
[]

First few duplicate unordered pairs:
[]

First few duplicate directed edges:
[]
Processing: ParseLLAMA_vifspdat.csv
num nodes: 30
num directed edges: 435
expected unordered edges: 435
actual unordered edges: 435
self loops: 0
missing unordered pairs: 0
duplicate unordered pairs: 0
duplicate directed edges: 0

First few missing pairs:
[]

First few duplicate unordered pairs:
[]

First few duplicate directed edges:
[]
Processing: Parseddeepseek8B_vifspdat.csv
num nodes: 30
num directed edges: 433
expected unordered edges: 435
actual unordered edges: 433
self loops: 0
missing unordered pairs: 2
duplicate unordered pairs: 0
duplicate directed edges: 0

First few missing pairs:
[(2398, 338275), (2398, 55440)]

First few duplicate unorde

,file,complete_check,naive_count,formula_count,num_nodes,expected_edges,actual_edges,num_missing_edges,missing_edges,num_duplicate_unordered_edges,num_duplicate_directed_edges,num_unknown_winners,unknown_winners
0,ParsedQWEN_vispdat.csv,passed,114,114,30,435,435,0,[],0,0,0,[]
1,ParseLLAMA_vifspdat.csv,passed,33,33,30,435,435,0,[],0,0,0,[]
2,Parseddeepseek8B_vifspdat.csv,failed,97,135,30,435,433,2,"[(2398, 338275), (2398, 55440)]",0,0,2,"[indeterminate, indeterminate]"


Consolidated code

In [35]:
import os
import pandas as pd
from collections import Counter
from evaluation.count_cycles import GraphTools

base_dir = r"D:\Vtech\vibeRank\responses"

file_configs = [
    # VISPDAT
    {
        "dataset": "VISPDAT",
        "model": "QWEN",
        "file": "QWEN_vispdat_20260428_133227.csv",
    },
    {
        "dataset": "VISPDAT",
        "model": "LLAMA",
        "file": "ParseLLAMA_vispdat_20260428_184821.csv",
    },
    {
        "dataset": "VISPDAT",
        "model": "DeepSeek8B",
        "file": "Parseddeepseek8B_vispdat_20260428_203611.csv",
    },

    # VIFSPDAT
    {
        "dataset": "VIFSPDAT",
        "model": "QWEN",
        "file": "ParsedQWEN_vispdat.csv",
    },
    {
        "dataset": "VIFSPDAT",
        "model": "LLAMA",
        "file": "ParseLLAMA_vifspdat.csv",
    },
    {
        "dataset": "VIFSPDAT",
        "model": "DeepSeek8B",
        "file": "Parseddeepseek8B_vifspdat.csv",
    },

    # TAYVISPDAT
    {
        "dataset": "TAYVISPDAT",
        "model": "QWEN",
        "file": "ParsedQWEN_TAY.csv",
    },
    {
        "dataset": "TAYVISPDAT",
        "model": "LLAMA",
        "file": "llama_TAY_parsed.csv",
    },
    {
        "dataset": "TAYVISPDAT",
        "model": "DeepSeek8B",
        "file": "DS_TAY_parsed.csv",
    },
]


def canonical_pair(u, v):
    return tuple(sorted([u, v], key=lambda x: str(x)))


def normalize_winner(x):
    x_raw = x

    if pd.isna(x):
        return "indeterminate"

    x = str(x).strip().lower()

    if x in ["household1", "household 1", "1", "left"]:
        return "household1"

    elif x in ["household2", "household 2", "2", "right"]:
        return "household2"

    elif x in [
        "indeterminate",
        "inderminate",
        "inderterminate",
        "undetermined",
        "unknown",
        "tie",
        "neither",
        "none",
        "nan",
    ]:
        return "indeterminate"

    else:
        return f"unknown::{x_raw}"


def choose_majority_winner(group):
    total_votes = len(group)

    counts = group["winner_clean"].value_counts(dropna=False)

    h1_votes = counts.get("household1", 0)
    h2_votes = counts.get("household2", 0)
    indeterminate_votes = counts.get("indeterminate", 0)

    unknown_votes = sum(
        count
        for label, count in counts.items()
        if str(label).startswith("unknown::")
    )

    # Rule:
    # Indeterminate can win only if ALL comparisons are indeterminate.
    if indeterminate_votes == total_votes:
        majority_winner = "all_indeterminate"
        majority_votes = indeterminate_votes
        edge_status = "all_indeterminate"

    else:
        # Otherwise ignore indeterminate and choose between household1 and household2.
        if h1_votes > h2_votes:
            majority_winner = "household1"
            majority_votes = h1_votes
            edge_status = "edge_created"

        elif h2_votes > h1_votes:
            majority_winner = "household2"
            majority_votes = h2_votes
            edge_status = "edge_created"

        else:
            # This means household1 and household2 are tied after ignoring indeterminate.
            majority_winner = "tie_household1_household2"
            majority_votes = h1_votes
            edge_status = "majority_tie"

    return pd.Series({
        "majority_winner": majority_winner,
        "edge_status": edge_status,
        "majority_votes": majority_votes,
        "total_votes": total_votes,
        "household1_votes": h1_votes,
        "household2_votes": h2_votes,
        "indeterminate_votes": indeterminate_votes,
        "unknown_votes": unknown_votes,
    })


def diagnose_edges(edges, all_nodes=None, verbose=False):
    nodes = set(all_nodes) if all_nodes is not None else set()

    unordered_pair_counts = Counter()
    directed_pair_counts = Counter()
    self_loops = []

    for u, v in edges:
        nodes.add(u)
        nodes.add(v)

        directed_pair_counts[(u, v)] += 1

        if u == v:
            self_loops.append((u, v))
        else:
            unordered_pair_counts[canonical_pair(u, v)] += 1

    n = len(nodes)
    expected_edges = n * (n - 1) // 2
    actual_edges = len(unordered_pair_counts)

    missing_pairs = []
    duplicate_unordered_pairs = []

    nodes_list = sorted(nodes, key=lambda x: str(x))

    for i in range(len(nodes_list)):
        for j in range(i + 1, len(nodes_list)):
            pair = canonical_pair(nodes_list[i], nodes_list[j])
            cnt = unordered_pair_counts.get(pair, 0)

            if cnt == 0:
                missing_pairs.append(pair)
            elif cnt > 1:
                duplicate_unordered_pairs.append((pair, cnt))

    duplicate_directed_edges = [
        (pair, cnt)
        for pair, cnt in directed_pair_counts.items()
        if cnt > 1
    ]

    if verbose:
        print("num nodes:", n)
        print("num directed edges:", len(edges))
        print("expected unordered edges:", expected_edges)
        print("actual unordered edges:", actual_edges)
        print("self loops:", len(self_loops))
        print("missing unordered pairs:", len(missing_pairs))
        print("duplicate unordered pairs:", len(duplicate_unordered_pairs))
        print("duplicate directed edges:", len(duplicate_directed_edges))

        print("\nFirst few missing pairs:")
        print(missing_pairs[:10])

        print("\nFirst few duplicate unordered pairs:")
        print(duplicate_unordered_pairs[:10])

        print("\nFirst few duplicate directed edges:")
        print(duplicate_directed_edges[:10])

    return {
        "num_nodes": n,
        "num_directed_edges": len(edges),
        "expected_edges": expected_edges,
        "actual_edges": actual_edges,
        "num_self_loops": len(self_loops),
        "num_missing_edges": len(missing_pairs),
        "missing_edges": missing_pairs,
        "num_duplicate_unordered_edges": len(duplicate_unordered_pairs),
        "duplicate_unordered_edges": duplicate_unordered_pairs,
        "num_duplicate_directed_edges": len(duplicate_directed_edges),
        "duplicate_directed_edges": duplicate_directed_edges,
    }


def build_majority_edges_from_file(dataset, model, file_name, verbose=False):
    t_path = os.path.join(base_dir, dataset, "rc_responses")
    file_path = os.path.join(t_path, file_name)

    df_parsed = pd.read_csv(file_path)
    df_parsed = df_parsed[["left_item", "right_item", "more_vulnerable_household"]].copy()

    all_nodes = set(df_parsed["left_item"]).union(set(df_parsed["right_item"]))

    df_tmp = df_parsed.copy()
    df_tmp["winner_clean"] = df_tmp["more_vulnerable_household"].apply(normalize_winner)

    df_majority = (
        df_tmp
        .groupby(["left_item", "right_item"])
        .apply(choose_majority_winner, include_groups=False)
        .reset_index()
    )

    edges = []

    majority_tie_edges = []
    all_indeterminate_edges = []
    unknown_edges = []

    for row in df_majority.itertuples(index=False):
        left = row.left_item
        right = row.right_item
        winner = row.majority_winner

        if winner == "household1":
            # left wins, so edge points right -> left
            edges.append((right, left))

        elif winner == "household2":
            # right wins, so edge points left -> right
            edges.append((left, right))

        elif winner == "tie_household1_household2":
            majority_tie_edges.append((left, right))

        elif winner == "all_indeterminate":
            all_indeterminate_edges.append((left, right))

        else:
            unknown_edges.append((left, right, winner))

    gt = GraphTools(edges)

    complete_check = gt.validate_complete_graph()

    try:
        formula_count = gt.count_triads()
    except Exception as e:
        formula_count = None
        if verbose:
            print("Formula count failed:", e)

    try:
        naive_count, cycles = gt.naive_count_triads()
    except Exception as e:
        naive_count = None
        cycles = None
        if verbose:
            print("Naive count failed:", e)

    diag = diagnose_edges(edges, all_nodes=all_nodes, verbose=verbose)

    formula_valid = complete_check == "passed"

    result = {
        "dataset": dataset,
        "model": model,
        "file": file_name,

        "complete_check": complete_check,
        "formula_valid": formula_valid,

        "naive_count": naive_count,
        "formula_count": formula_count,

        "num_nodes": diag["num_nodes"],
        "expected_edges": diag["expected_edges"],
        "actual_edges": diag["actual_edges"],
        "num_missing_edges": diag["num_missing_edges"],
        "missing_edges": diag["missing_edges"],

        "num_majority_tie_edges": len(majority_tie_edges),
        "majority_tie_edges": majority_tie_edges,

        "num_all_indeterminate_edges": len(all_indeterminate_edges),
        "all_indeterminate_edges": all_indeterminate_edges,

        "num_unknown_edges": len(unknown_edges),
        "unknown_edges": unknown_edges,

        "num_duplicate_unordered_edges": diag["num_duplicate_unordered_edges"],
        "duplicate_unordered_edges": diag["duplicate_unordered_edges"],

        "num_duplicate_directed_edges": diag["num_duplicate_directed_edges"],
        "duplicate_directed_edges": diag["duplicate_directed_edges"],

        "total_ordered_pairs_after_majority": len(df_majority),
        "total_edges_created": len(edges),
    }

    return result, df_majority, edges, cycles


all_results = []
all_majority_dfs = {}
all_edges = {}
all_cycles = {}

for config in file_configs:
    dataset = config["dataset"]
    model = config["model"]
    file_name = config["file"]

    print("=" * 100)
    print(f"Processing: dataset={dataset}, model={model}, file={file_name}")

    result, df_majority, edges, cycles = build_majority_edges_from_file(
        dataset=dataset,
        model=model,
        file_name=file_name,
        verbose=True,
    )

    key = (dataset, model, file_name)

    all_results.append(result)
    all_majority_dfs[key] = df_majority
    all_edges[key] = edges
    all_cycles[key] = cycles

summary_df = pd.DataFrame(all_results)

summary_cols = [
    "dataset",
    "model",
    "file",
    "complete_check",
    "formula_valid",
    "naive_count",
    "formula_count",
    "expected_edges",
    "actual_edges",
    "num_missing_edges",
    "missing_edges",
    "num_majority_tie_edges",
    "majority_tie_edges",
    "num_all_indeterminate_edges",
    "all_indeterminate_edges",
    "num_unknown_edges",
    "unknown_edges",
]

summary_df = summary_df[summary_cols]

summary_df

Processing: dataset=VISPDAT, model=QWEN, file=QWEN_vispdat_20260428_133227.csv
num nodes: 30
num directed edges: 434
expected unordered edges: 435
actual unordered edges: 434
self loops: 0
missing unordered pairs: 1
duplicate unordered pairs: 0
duplicate directed edges: 0

First few missing pairs:
[(218687, 337899)]

First few duplicate unordered pairs:
[]

First few duplicate directed edges:
[]
Processing: dataset=VISPDAT, model=LLAMA, file=ParseLLAMA_vispdat_20260428_184821.csv
num nodes: 30
num directed edges: 434
expected unordered edges: 435
actual unordered edges: 434
self loops: 0
missing unordered pairs: 1
duplicate unordered pairs: 0
duplicate directed edges: 0

First few missing pairs:
[(256880, 264338)]

First few duplicate unordered pairs:
[]

First few duplicate directed edges:
[]
Processing: dataset=VISPDAT, model=DeepSeek8B, file=Parseddeepseek8B_vispdat_20260428_203611.csv
num nodes: 30
num directed edges: 426
expected unordered edges: 435
actual unordered edges: 426
se

,dataset,model,file,complete_check,formula_valid,naive_count,formula_count,expected_edges,actual_edges,num_missing_edges,missing_edges,num_majority_tie_edges,majority_tie_edges,num_all_indeterminate_edges,all_indeterminate_edges,num_unknown_edges,unknown_edges
0,VISPDAT,QWEN,QWEN_vispdat_20260428_133227.csv,failed,False,48,52,435,434,1,"[(218687, 337899)]",1,"[(337899, 218687)]",0,[],0,[]
1,VISPDAT,LLAMA,ParseLLAMA_vispdat_20260428_184821.csv,failed,False,30,50,435,434,1,"[(256880, 264338)]",1,"[(264338, 256880)]",0,[],0,[]
2,VISPDAT,DeepSeek8B,Parseddeepseek8B_vispdat_20260428_203611.csv,failed,False,61,204,435,426,9,"[(18, 211435), (18, 262923), (18, 349157), (20...",9,"[(18, 349157), (84305, 230809), (207017, 34915...",0,[],0,[]
3,VIFSPDAT,QWEN,ParsedQWEN_vispdat.csv,passed,True,114,114,435,435,0,[],0,[],0,[],0,[]
4,VIFSPDAT,LLAMA,ParseLLAMA_vifspdat.csv,failed,False,33,54,435,432,3,"[(239880, 286810), (282044, 354439), (286810, ...",3,"[(282044, 354439), (286810, 239880), (286810, ...",0,[],0,[]
5,VIFSPDAT,DeepSeek8B,Parseddeepseek8B_vifspdat.csv,failed,False,89,244,435,427,8,"[(209255, 218663), (211417, 252791), (2398, 35...",8,"[(505, 286810), (2398, 354439), (55440, 291246...",0,[],0,[]
6,TAYVISPDAT,QWEN,ParsedQWEN_TAY.csv,failed,False,98,122,435,434,1,"[(292482, 354459)]",1,"[(292482, 354459)]",0,[],0,[]
7,TAYVISPDAT,LLAMA,llama_TAY_parsed.csv,passed,True,47,47,435,435,0,[],0,[],0,[],0,[]
8,TAYVISPDAT,DeepSeek8B,DS_TAY_parsed.csv,failed,False,117,175,435,431,4,"[(234040, 306436), (244505, 301098), (275125, ...",4,"[(53565, 298032), (234040, 306436), (301098, 2...",0,[],0,[]


randomly making one win


In [36]:
import os
import numpy as np
import pandas as pd
from collections import Counter

from evaluation.count_cycles import GraphTools


# =============================================================================
# Config
# =============================================================================

base_dir = r"D:\Vtech\vibeRank\responses"

file_configs = [
    # -------------------------------------------------------------------------
    # VISPDAT
    # -------------------------------------------------------------------------
    {
        "dataset": "VISPDAT",
        "model": "QWEN",
        "file": "QWEN_vispdat_20260428_133227.csv",
    },
    {
        "dataset": "VISPDAT",
        "model": "LLAMA",
        "file": "ParseLLAMA_vispdat_20260428_184821.csv",
    },
    {
        "dataset": "VISPDAT",
        "model": "DeepSeek8B",
        "file": "Parseddeepseek8B_vispdat_20260428_203611.csv",
    },

    # -------------------------------------------------------------------------
    # VIFSPDAT
    # -------------------------------------------------------------------------
    {
        "dataset": "VIFSPDAT",
        "model": "QWEN",
        "file": "ParsedQWEN_vispdat.csv",
    },
    {
        "dataset": "VIFSPDAT",
        "model": "LLAMA",
        "file": "ParseLLAMA_vifspdat.csv",
    },
    {
        "dataset": "VIFSPDAT",
        "model": "DeepSeek8B",
        "file": "Parseddeepseek8B_vifspdat.csv",
    },

    # -------------------------------------------------------------------------
    # TAYVISPDAT
    # -------------------------------------------------------------------------
    {
        "dataset": "TAYVISPDAT",
        "model": "QWEN",
        "file": "ParsedQWEN_TAY.csv",
    },
    {
        "dataset": "TAYVISPDAT",
        "model": "LLAMA",
        "file": "llama_TAY_parsed.csv",
    },
    {
        "dataset": "TAYVISPDAT",
        "model": "DeepSeek8B",
        "file": "DS_TAY_parsed.csv",
    },
]

RANDOM_SEED = 10


# =============================================================================
# Helpers
# =============================================================================

def canonical_pair(u, v):
    return tuple(sorted([u, v], key=lambda x: str(x)))


def normalize_winner(x):
    if pd.isna(x):
        return "indeterminate"

    x_raw = x
    x = str(x).strip().lower()

    if x in ["household1", "household 1", "1", "left"]:
        return "household1"

    if x in ["household2", "household 2", "2", "right"]:
        return "household2"

    if x in [
        "indeterminate",
        "inderminate",
        "inderterminate",
        "undetermined",
        "unknown",
        "tie",
        "neither",
        "none",
        "nan",
    ]:
        return "indeterminate"

    return f"unknown::{x_raw}"


def choose_majority_winner(group):
    total_votes = len(group)
    counts = group["winner_clean"].value_counts(dropna=False)

    h1_votes = counts.get("household1", 0)
    h2_votes = counts.get("household2", 0)
    indeterminate_votes = counts.get("indeterminate", 0)

    unknown_votes = sum(
        count
        for label, count in counts.items()
        if str(label).startswith("unknown::")
    )

    # Rule:
    # indeterminate can win only if ALL comparisons are indeterminate.
    if indeterminate_votes == total_votes:
        majority_winner = "all_indeterminate"
        majority_votes = indeterminate_votes
        edge_status = "all_indeterminate"

    else:
        # Otherwise ignore indeterminate and choose between household1 and household2.
        if h1_votes > h2_votes:
            majority_winner = "household1"
            majority_votes = h1_votes
            edge_status = "edge_created"

        elif h2_votes > h1_votes:
            majority_winner = "household2"
            majority_votes = h2_votes
            edge_status = "edge_created"

        else:
            # household1 and household2 tied after ignoring indeterminate.
            majority_winner = "tie_household1_household2"
            majority_votes = h1_votes
            edge_status = "majority_tie"

    return pd.Series({
        "majority_winner": majority_winner,
        "edge_status": edge_status,
        "majority_votes": majority_votes,
        "total_votes": total_votes,
        "household1_votes": h1_votes,
        "household2_votes": h2_votes,
        "indeterminate_votes": indeterminate_votes,
        "unknown_votes": unknown_votes,
    })


def diagnose_edges(edges, all_nodes=None, verbose=False):
    nodes = set(all_nodes) if all_nodes is not None else set()

    unordered_pair_counts = Counter()
    directed_pair_counts = Counter()
    self_loops = []

    for u, v in edges:
        nodes.add(u)
        nodes.add(v)

        directed_pair_counts[(u, v)] += 1

        if u == v:
            self_loops.append((u, v))
        else:
            unordered_pair_counts[canonical_pair(u, v)] += 1

    n = len(nodes)
    expected_edges = n * (n - 1) // 2
    actual_edges = len(unordered_pair_counts)

    missing_pairs = []
    duplicate_unordered_pairs = []

    nodes_list = sorted(nodes, key=lambda x: str(x))

    for i in range(len(nodes_list)):
        for j in range(i + 1, len(nodes_list)):
            pair = canonical_pair(nodes_list[i], nodes_list[j])
            cnt = unordered_pair_counts.get(pair, 0)

            if cnt == 0:
                missing_pairs.append(pair)
            elif cnt > 1:
                duplicate_unordered_pairs.append((pair, cnt))

    duplicate_directed_edges = [
        (pair, cnt)
        for pair, cnt in directed_pair_counts.items()
        if cnt > 1
    ]

    if verbose:
        print("num nodes:", n)
        print("num directed edges:", len(edges))
        print("expected unordered edges:", expected_edges)
        print("actual unordered edges:", actual_edges)
        print("self loops:", len(self_loops))
        print("missing unordered pairs:", len(missing_pairs))
        print("duplicate unordered pairs:", len(duplicate_unordered_pairs))
        print("duplicate directed edges:", len(duplicate_directed_edges))

        print("\nFirst few missing pairs:")
        print(missing_pairs[:10])

        print("\nFirst few duplicate unordered pairs:")
        print(duplicate_unordered_pairs[:10])

        print("\nFirst few duplicate directed edges:")
        print(duplicate_directed_edges[:10])

    return {
        "num_nodes": n,
        "num_directed_edges": len(edges),
        "expected_edges": expected_edges,
        "actual_edges": actual_edges,
        "num_self_loops": len(self_loops),
        "num_missing_edges": len(missing_pairs),
        "missing_edges": missing_pairs,
        "num_duplicate_unordered_edges": len(duplicate_unordered_pairs),
        "duplicate_unordered_edges": duplicate_unordered_pairs,
        "num_duplicate_directed_edges": len(duplicate_directed_edges),
        "duplicate_directed_edges": duplicate_directed_edges,
    }


def build_majority_edges_from_file(dataset, model, file_name, random_seed=10, verbose=False):
    t_path = os.path.join(base_dir, dataset, "rc_responses")
    file_path = os.path.join(t_path, file_name)

    df_parsed = pd.read_csv(file_path)

    df_parsed = df_parsed[
        ["left_item", "right_item", "more_vulnerable_household"]
    ].copy()

    all_nodes = set(df_parsed["left_item"]).union(set(df_parsed["right_item"]))

    df_tmp = df_parsed.copy()
    df_tmp["winner_clean"] = df_tmp["more_vulnerable_household"].apply(normalize_winner)

    # Majority winner per ordered left/right pair.
    df_majority = (
        df_tmp
        .groupby(["left_item", "right_item"])
        .apply(choose_majority_winner)
        .reset_index()
    )

    # -------------------------------------------------------------------------
    # Randomly resolve household1/household2 majority ties
    # -------------------------------------------------------------------------
    tie_mask = df_majority["majority_winner"] == "tie_household1_household2"

    majority_tie_edges_before_random = list(
        df_majority.loc[tie_mask, ["left_item", "right_item"]]
        .itertuples(index=False, name=None)
    )

    rng = np.random.default_rng(random_seed)

    if tie_mask.sum() > 0:
        random_tie_winners = rng.choice(
            ["household1", "household2"],
            size=tie_mask.sum(),
        )

        df_majority.loc[tie_mask, "majority_winner"] = random_tie_winners
        df_majority.loc[tie_mask, "edge_status"] = "edge_created_random_tie_break"

    df_majority["randomly_broke_tie"] = False
    df_majority.loc[tie_mask, "randomly_broke_tie"] = True

    # -------------------------------------------------------------------------
    # Build edges
    # Edge points toward winner:
    # household1 wins => right -> left
    # household2 wins => left -> right
    # -------------------------------------------------------------------------
    edges = []

    all_indeterminate_edges = []
    unknown_edges = []

    for row in df_majority.itertuples(index=False):
        left = row.left_item
        right = row.right_item
        winner = row.majority_winner

        if winner == "household1":
            edges.append((right, left))

        elif winner == "household2":
            edges.append((left, right))

        elif winner == "all_indeterminate":
            all_indeterminate_edges.append((left, right))

        else:
            unknown_edges.append((left, right, winner))

    # -------------------------------------------------------------------------
    # Count cycles
    # -------------------------------------------------------------------------
    gt = GraphTools(edges)

    complete_check = gt.validate_complete_graph()

    try:
        formula_count = gt.count_triads()
    except Exception as e:
        formula_count = None
        if verbose:
            print("Formula count failed:", e)

    try:
        naive_count, cycles = gt.naive_count_triads()
    except Exception as e:
        naive_count = None
        cycles = None
        if verbose:
            print("Naive count failed:", e)

    diag = diagnose_edges(edges, all_nodes=all_nodes, verbose=verbose)

    result = {
        "dataset": dataset,
        "model": model,
        "file": file_name,

        "complete_check": complete_check,
        "formula_valid": complete_check == "passed",

        "naive_count": naive_count,
        "formula_count": formula_count,

        "num_nodes": diag["num_nodes"],
        "expected_edges": diag["expected_edges"],
        "actual_edges": diag["actual_edges"],

        "num_missing_edges": diag["num_missing_edges"],
        "missing_edges": diag["missing_edges"],

        "num_majority_tie_edges": len(majority_tie_edges_before_random),
        "majority_tie_edges": majority_tie_edges_before_random,
        "num_majority_tie_edges_randomly_resolved": len(majority_tie_edges_before_random),

        "num_all_indeterminate_edges": len(all_indeterminate_edges),
        "all_indeterminate_edges": all_indeterminate_edges,

        "num_unknown_edges": len(unknown_edges),
        "unknown_edges": unknown_edges,

        "num_duplicate_unordered_edges": diag["num_duplicate_unordered_edges"],
        "duplicate_unordered_edges": diag["duplicate_unordered_edges"],

        "num_duplicate_directed_edges": diag["num_duplicate_directed_edges"],
        "duplicate_directed_edges": diag["duplicate_directed_edges"],

        "total_ordered_pairs_after_majority": len(df_majority),
        "total_edges_created": len(edges),
    }

    return result, df_majority, edges, cycles


# =============================================================================
# Run all 9 files
# =============================================================================

all_results = []
all_majority_dfs = {}
all_edges = {}
all_cycles = {}

for config in file_configs:
    dataset = config["dataset"]
    model = config["model"]
    file_name = config["file"]

    print("=" * 100)
    print(f"Processing: dataset={dataset}, model={model}, file={file_name}")

    result, df_majority, edges, cycles = build_majority_edges_from_file(
        dataset=dataset,
        model=model,
        file_name=file_name,
        random_seed=RANDOM_SEED,
        verbose=True,
    )

    key = (dataset, model, file_name)

    all_results.append(result)
    all_majority_dfs[key] = df_majority
    all_edges[key] = edges
    all_cycles[key] = cycles


# =============================================================================
# Final summary table
# =============================================================================

summary_df = pd.DataFrame(all_results)

summary_cols = [
    "dataset",
    "model",
    "file",

    "complete_check",
    "formula_valid",

    "naive_count",
    "formula_count",

    "num_nodes",
    "expected_edges",
    "actual_edges",

    "num_missing_edges",
    "missing_edges",

    "num_majority_tie_edges",
    "num_majority_tie_edges_randomly_resolved",
    "majority_tie_edges",

    "num_all_indeterminate_edges",
    "all_indeterminate_edges",

    "num_unknown_edges",
    "unknown_edges",

    "total_ordered_pairs_after_majority",
    "total_edges_created",
]

summary_df = summary_df[summary_cols]

summary_df

Processing: dataset=VISPDAT, model=QWEN, file=QWEN_vispdat_20260428_133227.csv


C:\Users\ShafkatFarabi\AppData\Local\Temp\ipykernel_3840\2413751528.py:261: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(choose_majority_winner)


num nodes: 30
num directed edges: 435
expected unordered edges: 435
actual unordered edges: 435
self loops: 0
missing unordered pairs: 0
duplicate unordered pairs: 0
duplicate directed edges: 0

First few missing pairs:
[]

First few duplicate unordered pairs:
[]

First few duplicate directed edges:
[]
Processing: dataset=VISPDAT, model=LLAMA, file=ParseLLAMA_vispdat_20260428_184821.csv
num nodes: 30
num directed edges: 435
expected unordered edges: 435
actual unordered edges: 435
self loops: 0
missing unordered pairs: 0
duplicate unordered pairs: 0
duplicate directed edges: 0

First few missing pairs:
[]

First few duplicate unordered pairs:
[]

First few duplicate directed edges:
[]
Processing: dataset=VISPDAT, model=DeepSeek8B, file=Parseddeepseek8B_vispdat_20260428_203611.csv


C:\Users\ShafkatFarabi\AppData\Local\Temp\ipykernel_3840\2413751528.py:261: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(choose_majority_winner)
C:\Users\ShafkatFarabi\AppData\Local\Temp\ipykernel_3840\2413751528.py:261: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(choose_majority_winner)


num nodes: 30
num directed edges: 435
expected unordered edges: 435
actual unordered edges: 435
self loops: 0
missing unordered pairs: 0
duplicate unordered pairs: 0
duplicate directed edges: 0

First few missing pairs:
[]

First few duplicate unordered pairs:
[]

First few duplicate directed edges:
[]
Processing: dataset=VIFSPDAT, model=QWEN, file=ParsedQWEN_vispdat.csv


...

num nodes: 30
num directed edges: 435
expected unordered edges: 435
actual unordered edges: 435
self loops: 0
missing unordered pairs: 0
duplicate unordered pairs: 0
duplicate directed edges: 0

First few missing pairs:
[]

First few duplicate unordered pairs:
[]

First few duplicate directed edges:
[]
Processing: dataset=TAYVISPDAT, model=LLAMA, file=llama_TAY_parsed.csv


C:\Users\ShafkatFarabi\AppData\Local\Temp\ipykernel_3840\2413751528.py:261: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(choose_majority_winner)


num nodes: 30
num directed edges: 435
expected unordered edges: 435
actual unordered edges: 435
self loops: 0
missing unordered pairs: 0
duplicate unordered pairs: 0
duplicate directed edges: 0

First few missing pairs:
[]

First few duplicate unordered pairs:
[]

First few duplicate directed edges:
[]
Processing: dataset=TAYVISPDAT, model=DeepSeek8B, file=DS_TAY_parsed.csv
num nodes: 30
num directed edges: 435
expected unordered edges: 435
actual unordered edges: 435
self loops: 0
missing unordered pairs: 0
duplicate unordered pairs: 0
duplicate directed edges: 0

First few missing pairs:
[]

First few duplicate unordered pairs:
[]

First few duplicate directed edges:
[]


C:\Users\ShafkatFarabi\AppData\Local\Temp\ipykernel_3840\2413751528.py:261: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(choose_majority_winner)


,dataset,model,file,complete_check,formula_valid,naive_count,formula_count,num_nodes,expected_edges,actual_edges,...,missing_edges,num_majority_tie_edges,num_majority_tie_edges_randomly_resolved,majority_tie_edges,num_all_indeterminate_edges,all_indeterminate_edges,num_unknown_edges,unknown_edges,total_ordered_pairs_after_majority,total_edges_created
0,VISPDAT,QWEN,QWEN_vispdat_20260428_133227.csv,passed,True,48,48,30,435,435,...,[],1,1,"[(337899, 218687)]",0,[],0,[],435,435
1,VISPDAT,LLAMA,ParseLLAMA_vispdat_20260428_184821.csv,passed,True,37,37,30,435,435,...,[],1,1,"[(264338, 256880)]",0,[],0,[],435,435
2,VISPDAT,DeepSeek8B,Parseddeepseek8B_vispdat_20260428_203611.csv,passed,True,74,74,30,435,435,...,[],9,9,"[(18, 349157), (84305, 230809), (207017, 34915...",0,[],0,[],435,435
3,VIFSPDAT,QWEN,ParsedQWEN_vispdat.csv,passed,True,114,114,30,435,435,...,[],0,0,[],0,[],0,[],435,435
4,VIFSPDAT,LLAMA,ParseLLAMA_vifspdat.csv,passed,True,34,34,30,435,435,...,[],3,3,"[(282044, 354439), (286810, 239880), (286810, ...",0,[],0,[],435,435
5,VIFSPDAT,DeepSeek8B,Parseddeepseek8B_vifspdat.csv,passed,True,113,113,30,435,435,...,[],8,8,"[(505, 286810), (2398, 354439), (55440, 291246...",0,[],0,[],435,435
6,TAYVISPDAT,QWEN,ParsedQWEN_TAY.csv,passed,True,100,100,30,435,435,...,[],1,1,"[(292482, 354459)]",0,[],0,[],435,435
7,TAYVISPDAT,LLAMA,llama_TAY_parsed.csv,passed,True,47,47,30,435,435,...,[],0,0,[],0,[],0,[],435,435
8,TAYVISPDAT,DeepSeek8B,DS_TAY_parsed.csv,passed,True,123,123,30,435,435,...,[],4,4,"[(53565, 298032), (234040, 306436), (301098, 2...",0,[],0,[],435,435


In [37]:
import os
import pandas as pd

base_dir = r"D:\Vtech\vibeRank\responses"

file_configs = [
    {"dataset": "VISPDAT", "model": "QWEN", "file": "QWEN_vispdat_20260428_133227.csv"},
    {"dataset": "VISPDAT", "model": "LLAMA", "file": "ParseLLAMA_vispdat_20260428_184821.csv"},
    {"dataset": "VISPDAT", "model": "DeepSeek8B", "file": "Parseddeepseek8B_vispdat_20260428_203611.csv"},

    {"dataset": "VIFSPDAT", "model": "QWEN", "file": "ParsedQWEN_vispdat.csv"},
    {"dataset": "VIFSPDAT", "model": "LLAMA", "file": "ParseLLAMA_vifspdat.csv"},
    {"dataset": "VIFSPDAT", "model": "DeepSeek8B", "file": "Parseddeepseek8B_vifspdat.csv"},

    {"dataset": "TAYVISPDAT", "model": "QWEN", "file": "ParsedQWEN_TAY.csv"},
    {"dataset": "TAYVISPDAT", "model": "LLAMA", "file": "llama_TAY_parsed.csv"},
    {"dataset": "TAYVISPDAT", "model": "DeepSeek8B", "file": "DS_TAY_parsed.csv"},
]


def normalize_winner(x):
    if pd.isna(x):
        return "indeterminate"

    x_raw = x
    x = str(x).strip().lower()

    if x in ["household1", "household 1", "1", "left"]:
        return "household1"

    if x in ["household2", "household 2", "2", "right"]:
        return "household2"

    if x in [
        "indeterminate",
        "inderminate",
        "inderterminate",
        "undetermined",
        "unknown",
        "tie",
        "neither",
        "none",
        "nan",
    ]:
        return "indeterminate"

    return f"unknown::{x_raw}"


violation_records = []

for config in file_configs:
    dataset = config["dataset"]
    model = config["model"]
    file_name = config["file"]

    file_path = os.path.join(base_dir, dataset, "rc_responses", file_name)

    df = pd.read_csv(file_path)
    df = df[["left_item", "right_item", "more_vulnerable_household"]].copy()
    df["winner_clean"] = df["more_vulnerable_household"].apply(normalize_winner)

    for (left, right), group in df.groupby(["left_item", "right_item"]):
        counts = group["winner_clean"].value_counts(dropna=False)

        total_votes = len(group)
        h1_votes = counts.get("household1", 0)
        h2_votes = counts.get("household2", 0)
        ind_votes = counts.get("indeterminate", 0)
        unknown_votes = sum(
            count
            for label, count in counts.items()
            if str(label).startswith("unknown::")
        )

        violation_type = None

        if ind_votes == total_votes:
            violation_type = "all_indeterminate"

        elif h1_votes == h2_votes:
            violation_type = "household1_household2_tie"

        elif unknown_votes > 0:
            violation_type = "unknown_winner_value"

        if violation_type is not None:
            violation_records.append({
                "dataset": dataset,
                "model": model,
                "file": file_name,
                "left_item": left,
                "right_item": right,
                "violation_type": violation_type,
                "total_votes": total_votes,
                "household1_votes": h1_votes,
                "household2_votes": h2_votes,
                "indeterminate_votes": ind_votes,
                "unknown_votes": unknown_votes,
                "raw_values": group["more_vulnerable_household"].tolist(),
            })

violations_df = pd.DataFrame(violation_records)

violations_df

,dataset,model,file,left_item,right_item,violation_type,total_votes,household1_votes,household2_votes,indeterminate_votes,unknown_votes,raw_values
0,VISPDAT,QWEN,QWEN_vispdat_20260428_133227.csv,337899,218687,household1_household2_tie,10,5,5,0,0,"[Household 2, Household 1, Household 1, Househ..."
1,VISPDAT,LLAMA,ParseLLAMA_vispdat_20260428_184821.csv,264338,256880,household1_household2_tie,10,5,5,0,0,"[Household 1, Household 2, Household 1, Househ..."
2,VISPDAT,DeepSeek8B,Parseddeepseek8B_vispdat_20260428_203611.csv,18,349157,household1_household2_tie,10,5,5,0,0,"[Household 1, Household 2, Household 1, Househ..."
3,VISPDAT,DeepSeek8B,Parseddeepseek8B_vispdat_20260428_203611.csv,84305,230809,household1_household2_tie,10,5,5,0,0,"[Household 1, Household 2, Household 2, Househ..."
4,VISPDAT,DeepSeek8B,Parseddeepseek8B_vispdat_20260428_203611.csv,207017,349157,household1_household2_tie,10,5,5,0,0,"[Household 2, Household 2, Household 1, Househ..."
5,VISPDAT,DeepSeek8B,Parseddeepseek8B_vispdat_20260428_203611.csv,211435,18,household1_household2_tie,10,5,5,0,0,"[Household 2, Household 2, Household 1, Househ..."
6,VISPDAT,DeepSeek8B,Parseddeepseek8B_vispdat_20260428_203611.csv,262923,18,household1_household2_tie,10,5,5,0,0,"[Household 1, Household 2, Household 2, Househ..."
7,VISPDAT,DeepSeek8B,Parseddeepseek8B_vispdat_20260428_203611.csv,325477,218687,household1_household2_tie,10,5,5,0,0,"[Household 2, Household 1, Household 1, Househ..."
8,VISPDAT,DeepSeek8B,Parseddeepseek8B_vispdat_20260428_203611.csv,331859,289226,household1_household2_tie,10,5,5,0,0,"[Household 2, Household 2, Household 2, Househ..."
9,VISPDAT,DeepSeek8B,Parseddeepseek8B_vispdat_20260428_203611.csv,353058,51071,household1_household2_tie,10,5,5,0,0,"[Household 2, Household 2, Household 1, Househ..."


In [38]:
import os
import numpy as np
import pandas as pd
from collections import Counter

from evaluation.count_cycles import GraphTools


# =============================================================================
# Config
# =============================================================================

base_dir = r"D:\Vtech\vibeRank\responses"
RANDOM_SEED = 10

file_configs = [
    # VISPDAT
    {"dataset": "VISPDAT", "model": "QWEN", "file": "QWEN_vispdat_20260428_133227.csv"},
    {"dataset": "VISPDAT", "model": "LLAMA", "file": "ParseLLAMA_vispdat_20260428_184821.csv"},
    {"dataset": "VISPDAT", "model": "DeepSeek8B", "file": "Parseddeepseek8B_vispdat_20260428_203611.csv"},

    # VIFSPDAT
    {"dataset": "VIFSPDAT", "model": "QWEN", "file": "ParsedQWEN_vispdat.csv"},
    {"dataset": "VIFSPDAT", "model": "LLAMA", "file": "ParseLLAMA_vifspdat.csv"},
    {"dataset": "VIFSPDAT", "model": "DeepSeek8B", "file": "Parseddeepseek8B_vifspdat.csv"},

    # TAYVISPDAT
    {"dataset": "TAYVISPDAT", "model": "QWEN", "file": "ParsedQWEN_TAY.csv"},
    {"dataset": "TAYVISPDAT", "model": "LLAMA", "file": "llama_TAY_parsed.csv"},
    {"dataset": "TAYVISPDAT", "model": "DeepSeek8B", "file": "DS_TAY_parsed.csv"},
]


# =============================================================================
# Shared helpers
# =============================================================================

def normalize_winner(x):
    if pd.isna(x):
        return "indeterminate"

    x_raw = x
    x = str(x).strip().lower()

    if x in ["household1", "household 1", "1", "left"]:
        return "household1"

    if x in ["household2", "household 2", "2", "right"]:
        return "household2"

    if x in [
        "indeterminate",
        "inderminate",
        "inderterminate",
        "undetermined",
        "unknown",
        "tie",
        "neither",
        "none",
        "nan",
    ]:
        return "indeterminate"

    return f"unknown::{x_raw}"


def canonical_pair(u, v):
    return tuple(sorted([u, v], key=lambda x: str(x)))


def add_edge_from_winner(edges, left, right, winner):
    if winner == "household1":
        # left wins, edge points right -> left
        edges.append((right, left))

    elif winner == "household2":
        # right wins, edge points left -> right
        edges.append((left, right))


def diagnose_edges(edges, all_nodes=None):
    nodes = set(all_nodes) if all_nodes is not None else set()

    unordered_pair_counts = Counter()
    directed_pair_counts = Counter()
    self_loops = []

    for u, v in edges:
        nodes.add(u)
        nodes.add(v)
        directed_pair_counts[(u, v)] += 1

        if u == v:
            self_loops.append((u, v))
        else:
            unordered_pair_counts[canonical_pair(u, v)] += 1

    n = len(nodes)
    expected_edges = n * (n - 1) // 2
    actual_edges = len(unordered_pair_counts)

    missing_edges = []
    duplicate_unordered_edges = []

    nodes_list = sorted(nodes, key=lambda x: str(x))

    for i in range(len(nodes_list)):
        for j in range(i + 1, len(nodes_list)):
            pair = canonical_pair(nodes_list[i], nodes_list[j])
            cnt = unordered_pair_counts.get(pair, 0)

            if cnt == 0:
                missing_edges.append(pair)
            elif cnt > 1:
                duplicate_unordered_edges.append((pair, cnt))

    duplicate_directed_edges = [
        (pair, cnt)
        for pair, cnt in directed_pair_counts.items()
        if cnt > 1
    ]

    return {
        "num_nodes": n,
        "expected_edges": expected_edges,
        "actual_edges": actual_edges,
        "num_missing_edges": len(missing_edges),
        "missing_edges": missing_edges,
        "num_duplicate_unordered_edges": len(duplicate_unordered_edges),
        "duplicate_unordered_edges": duplicate_unordered_edges,
        "num_duplicate_directed_edges": len(duplicate_directed_edges),
        "duplicate_directed_edges": duplicate_directed_edges,
        "num_self_loops": len(self_loops),
        "self_loops": self_loops,
    }


def count_cycles_from_edges(edges, all_nodes=None):
    gt = GraphTools(edges)

    complete_check = gt.validate_complete_graph()

    try:
        formula_count = gt.count_triads()
    except Exception:
        formula_count = None

    try:
        naive_count, cycles = gt.naive_count_triads()
    except Exception:
        naive_count = None
        cycles = None

    diag = diagnose_edges(edges, all_nodes=all_nodes)

    return {
        "complete_check": complete_check,
        "formula_valid": complete_check == "passed",
        "naive_count": naive_count,
        "formula_count": formula_count,
        **diag,
    }, cycles


def load_one_file(dataset, model, file_name):
    file_path = os.path.join(base_dir, dataset, "rc_responses", file_name)

    df = pd.read_csv(file_path)
    df = df[["left_item", "right_item", "more_vulnerable_household"]].copy()

    df["dataset"] = dataset
    df["model"] = model
    df["file"] = file_name
    df["winner_clean"] = df["more_vulnerable_household"].apply(normalize_winner)
    df["row_order"] = np.arange(len(df))

    return df


# =============================================================================
# VERSION 1:
# For each file, for each left/right pair,
# take the FIRST non-indeterminate comparison as winner.
# =============================================================================

def build_first_non_indeterminate_edges_for_file(dataset, model, file_name):
    df = load_one_file(dataset, model, file_name)

    all_nodes = set(df["left_item"]).union(set(df["right_item"]))

    selected_rows = []
    all_indeterminate_pairs = []
    unknown_pairs = []

    for (left, right), group in df.groupby(["left_item", "right_item"], sort=False):
        group = group.sort_values("row_order")

        valid = group[group["winner_clean"].isin(["household1", "household2"])]

        if len(valid) > 0:
            chosen = valid.iloc[0].copy()
            selected_rows.append(chosen)

        else:
            unknown = group[group["winner_clean"].astype(str).str.startswith("unknown::")]

            if len(unknown) > 0:
                unknown_pairs.append((left, right))
            else:
                all_indeterminate_pairs.append((left, right))

    df_selected = pd.DataFrame(selected_rows)

    edges = []

    for row in df_selected.itertuples(index=False):
        add_edge_from_winner(
            edges=edges,
            left=row.left_item,
            right=row.right_item,
            winner=row.winner_clean,
        )

    count_result, cycles = count_cycles_from_edges(edges, all_nodes=all_nodes)

    result = {
        "version": "first_non_indeterminate",
        "dataset": dataset,
        "model": model,
        "file": file_name,

        "naive_count": count_result["naive_count"],
        "formula_count": count_result["formula_count"],
        "complete_check": count_result["complete_check"],
        "formula_valid": count_result["formula_valid"],

        "num_nodes": count_result["num_nodes"],
        "expected_edges": count_result["expected_edges"],
        "actual_edges": count_result["actual_edges"],
        "num_missing_edges": count_result["num_missing_edges"],
        "missing_edges": count_result["missing_edges"],

        "num_all_indeterminate_pairs": len(all_indeterminate_pairs),
        "all_indeterminate_pairs": all_indeterminate_pairs,

        "num_unknown_pairs": len(unknown_pairs),
        "unknown_pairs": unknown_pairs,

        "num_edges_created": len(edges),
    }

    return result, df_selected, edges, cycles


first_non_indeterminate_results = []
first_non_indeterminate_selected = {}
first_non_indeterminate_edges = {}
first_non_indeterminate_cycles = {}

for config in file_configs:
    dataset = config["dataset"]
    model = config["model"]
    file_name = config["file"]

    result, df_selected, edges, cycles = build_first_non_indeterminate_edges_for_file(
        dataset=dataset,
        model=model,
        file_name=file_name,
    )

    key = (dataset, model, file_name)

    first_non_indeterminate_results.append(result)
    first_non_indeterminate_selected[key] = df_selected
    first_non_indeterminate_edges[key] = edges
    first_non_indeterminate_cycles[key] = cycles

first_non_indeterminate_summary_df = pd.DataFrame(first_non_indeterminate_results)

first_non_indeterminate_summary_df

,version,dataset,model,file,naive_count,formula_count,complete_check,formula_valid,num_nodes,expected_edges,actual_edges,num_missing_edges,missing_edges,num_all_indeterminate_pairs,all_indeterminate_pairs,num_unknown_pairs,unknown_pairs,num_edges_created
0,first_non_indeterminate,VISPDAT,QWEN,QWEN_vispdat_20260428_133227.csv,52,52,passed,True,30,435,435,0,[],0,[],0,[],435
1,first_non_indeterminate,VISPDAT,LLAMA,ParseLLAMA_vispdat_20260428_184821.csv,41,41,passed,True,30,435,435,0,[],0,[],0,[],435
2,first_non_indeterminate,VISPDAT,DeepSeek8B,Parseddeepseek8B_vispdat_20260428_203611.csv,207,207,passed,True,30,435,435,0,[],0,[],0,[],435
3,first_non_indeterminate,VIFSPDAT,QWEN,ParsedQWEN_vispdat.csv,116,116,passed,True,30,435,435,0,[],0,[],0,[],435
4,first_non_indeterminate,VIFSPDAT,LLAMA,ParseLLAMA_vifspdat.csv,95,95,passed,True,30,435,435,0,[],0,[],0,[],435
5,first_non_indeterminate,VIFSPDAT,DeepSeek8B,Parseddeepseek8B_vifspdat.csv,125,125,passed,True,30,435,435,0,[],0,[],0,[],435
6,first_non_indeterminate,TAYVISPDAT,QWEN,ParsedQWEN_TAY.csv,100,100,passed,True,30,435,435,0,[],0,[],0,[],435
7,first_non_indeterminate,TAYVISPDAT,LLAMA,llama_TAY_parsed.csv,40,40,passed,True,30,435,435,0,[],0,[],0,[],435
8,first_non_indeterminate,TAYVISPDAT,DeepSeek8B,DS_TAY_parsed.csv,116,116,passed,True,30,435,435,0,[],0,[],0,[],435


In [39]:
# =============================================================================
# VERSION 2:
# For each dataset, pool all 3 model files together.
# For each left/right pair, take majority across all pooled comparisons.
# Indeterminate cannot win unless ALL comparisons are indeterminate.
# If household1 and household2 tie, randomly break tie.
# =============================================================================

def choose_pooled_majority_winner(group, rng):
    total_votes = len(group)
    counts = group["winner_clean"].value_counts(dropna=False)

    h1_votes = counts.get("household1", 0)
    h2_votes = counts.get("household2", 0)
    indeterminate_votes = counts.get("indeterminate", 0)

    unknown_votes = sum(
        count
        for label, count in counts.items()
        if str(label).startswith("unknown::")
    )

    if indeterminate_votes == total_votes:
        majority_winner = "all_indeterminate"
        edge_status = "all_indeterminate"
        majority_votes = indeterminate_votes
        randomly_broke_tie = False

    else:
        if h1_votes > h2_votes:
            majority_winner = "household1"
            edge_status = "edge_created"
            majority_votes = h1_votes
            randomly_broke_tie = False

        elif h2_votes > h1_votes:
            majority_winner = "household2"
            edge_status = "edge_created"
            majority_votes = h2_votes
            randomly_broke_tie = False

        else:
            majority_winner = rng.choice(["household1", "household2"])
            edge_status = "edge_created_random_tie_break"
            majority_votes = h1_votes
            randomly_broke_tie = True

    return pd.Series({
        "majority_winner": majority_winner,
        "edge_status": edge_status,
        "majority_votes": majority_votes,
        "total_votes": total_votes,
        "household1_votes": h1_votes,
        "household2_votes": h2_votes,
        "indeterminate_votes": indeterminate_votes,
        "unknown_votes": unknown_votes,
        "randomly_broke_tie": randomly_broke_tie,
    })


def build_pooled_3_model_edges_for_dataset(dataset, random_seed=10):
    rng = np.random.default_rng(random_seed)

    dataset_configs = [
        config
        for config in file_configs
        if config["dataset"] == dataset
    ]

    dfs = []

    for config in dataset_configs:
        df_one = load_one_file(
            dataset=config["dataset"],
            model=config["model"],
            file_name=config["file"],
        )
        dfs.append(df_one)

    df_pooled = pd.concat(dfs, ignore_index=True)

    all_nodes = set(df_pooled["left_item"]).union(set(df_pooled["right_item"]))

    majority_rows = []

    for (left, right), group in df_pooled.groupby(["left_item", "right_item"], sort=False):
        stats = choose_pooled_majority_winner(group, rng)

        row = {
            "dataset": dataset,
            "left_item": left,
            "right_item": right,
            **stats.to_dict(),
        }

        majority_rows.append(row)

    df_pooled_majority = pd.DataFrame(majority_rows)

    edges = []
    all_indeterminate_pairs = []
    random_tie_pairs = []
    unknown_pairs = []

    for row in df_pooled_majority.itertuples(index=False):
        left = row.left_item
        right = row.right_item
        winner = row.majority_winner

        if winner in ["household1", "household2"]:
            add_edge_from_winner(
                edges=edges,
                left=left,
                right=right,
                winner=winner,
            )

            if row.randomly_broke_tie:
                random_tie_pairs.append((left, right))

        elif winner == "all_indeterminate":
            all_indeterminate_pairs.append((left, right))

        else:
            unknown_pairs.append((left, right, winner))

    count_result, cycles = count_cycles_from_edges(edges, all_nodes=all_nodes)

    result = {
        "version": "pooled_3_models_majority",
        "dataset": dataset,
        "model": "POOLED_3_MODELS",
        "file": "POOLED_3_MODEL_FILES",

        "naive_count": count_result["naive_count"],
        "formula_count": count_result["formula_count"],
        "complete_check": count_result["complete_check"],
        "formula_valid": count_result["formula_valid"],

        "num_nodes": count_result["num_nodes"],
        "expected_edges": count_result["expected_edges"],
        "actual_edges": count_result["actual_edges"],
        "num_missing_edges": count_result["num_missing_edges"],
        "missing_edges": count_result["missing_edges"],

        "num_random_tie_pairs": len(random_tie_pairs),
        "random_tie_pairs": random_tie_pairs,

        "num_all_indeterminate_pairs": len(all_indeterminate_pairs),
        "all_indeterminate_pairs": all_indeterminate_pairs,

        "num_unknown_pairs": len(unknown_pairs),
        "unknown_pairs": unknown_pairs,

        "num_edges_created": len(edges),
        "num_pooled_rows": len(df_pooled),
    }

    return result, df_pooled_majority, df_pooled, edges, cycles


pooled_results = []
pooled_majority_dfs = {}
pooled_raw_dfs = {}
pooled_edges = {}
pooled_cycles = {}

for dataset in ["VISPDAT", "VIFSPDAT", "TAYVISPDAT"]:
    result, df_pooled_majority, df_pooled, edges, cycles = build_pooled_3_model_edges_for_dataset(
        dataset=dataset,
        random_seed=RANDOM_SEED,
    )

    pooled_results.append(result)
    pooled_majority_dfs[dataset] = df_pooled_majority
    pooled_raw_dfs[dataset] = df_pooled
    pooled_edges[dataset] = edges
    pooled_cycles[dataset] = cycles

pooled_3_models_summary_df = pd.DataFrame(pooled_results)

pooled_3_models_summary_df

,version,dataset,model,file,naive_count,formula_count,complete_check,formula_valid,num_nodes,expected_edges,...,num_missing_edges,missing_edges,num_random_tie_pairs,random_tie_pairs,num_all_indeterminate_pairs,all_indeterminate_pairs,num_unknown_pairs,unknown_pairs,num_edges_created,num_pooled_rows
0,pooled_3_models_majority,VISPDAT,POOLED_3_MODELS,POOLED_3_MODEL_FILES,52,52,passed,True,30,435,...,0,[],5,"[(588, 284566), (353058, 234634), (325477, 218...",0,[],0,[],435,13048
1,pooled_3_models_majority,VIFSPDAT,POOLED_3_MODELS,POOLED_3_MODEL_FILES,53,53,passed,True,30,435,...,0,[],3,"[(355004, 336774), (286810, 239880), (55440, 3...",0,[],0,[],435,13050
2,pooled_3_models_majority,TAYVISPDAT,POOLED_3_MODELS,POOLED_3_MODEL_FILES,37,37,passed,True,30,435,...,0,[],4,"[(301098, 275125), (53565, 298032), (338127, 3...",0,[],0,[],435,13050


In [40]:
# Net change in naive cycle counts:
# first_non_indeterminate_summary_df -> summary_df -> pooled_3_models_summary_df

cycle_col = "naive_count"

first_df = first_non_indeterminate_summary_df[
    ["dataset", "model", "file", cycle_col]
].rename(columns={cycle_col: "first_non_indeterminate_count"})

majority_df = summary_df[
    ["dataset", "model", "file", cycle_col]
].rename(columns={cycle_col: "majority_count"})

pooled_df = pooled_3_models_summary_df[
    ["dataset", cycle_col]
].rename(columns={cycle_col: "pooled_3_models_count"})

net_change_df = (
    first_df
    .merge(
        majority_df,
        on=["dataset", "model", "file"],
        how="left"
    )
    .merge(
        pooled_df,
        on="dataset",
        how="left"
    )
)

net_change_df["change_first_to_majority"] = (
    net_change_df["majority_count"]
    - net_change_df["first_non_indeterminate_count"]
)

net_change_df["change_majority_to_pooled"] = (
    net_change_df["pooled_3_models_count"]
    - net_change_df["majority_count"]
)

net_change_df["net_change_first_to_pooled"] = (
    net_change_df["pooled_3_models_count"]
    - net_change_df["first_non_indeterminate_count"]
)

net_change_df

,dataset,model,file,first_non_indeterminate_count,majority_count,pooled_3_models_count,change_first_to_majority,change_majority_to_pooled,net_change_first_to_pooled
0,VISPDAT,QWEN,QWEN_vispdat_20260428_133227.csv,52,48,52,-4,4,0
1,VISPDAT,LLAMA,ParseLLAMA_vispdat_20260428_184821.csv,41,37,52,-4,15,11
2,VISPDAT,DeepSeek8B,Parseddeepseek8B_vispdat_20260428_203611.csv,207,74,52,-133,-22,-155
3,VIFSPDAT,QWEN,ParsedQWEN_vispdat.csv,116,114,53,-2,-61,-63
4,VIFSPDAT,LLAMA,ParseLLAMA_vifspdat.csv,95,34,53,-61,19,-42
5,VIFSPDAT,DeepSeek8B,Parseddeepseek8B_vifspdat.csv,125,113,53,-12,-60,-72
6,TAYVISPDAT,QWEN,ParsedQWEN_TAY.csv,100,100,37,0,-63,-63
7,TAYVISPDAT,LLAMA,llama_TAY_parsed.csv,40,47,37,7,-10,-3
8,TAYVISPDAT,DeepSeek8B,DS_TAY_parsed.csv,116,123,37,7,-86,-79
